In [1]:
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import minmax_scale
import copy
import os 

dtype = torch.float
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
class FCN(nn.Module):
    def __init__(self, input_size, hidden_size=64, output_size=1):
        super(FCN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, hidden_size)
        self.fc4 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(p=0.4)
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc4(x)
        return x

In [ ]:
def plot_roc_curve(y_true, y_score):
    y_pred = np.where(y_score >= 0.5, 1, 0)
    tp = np.count_nonzero(np.logical_and(y_pred == 1, y_true == 1))
    tn = np.count_nonzero(np.logical_and(y_pred == 0, y_true == 0))
    fp = np.count_nonzero(np.logical_and(y_pred == 1, y_true == 0))
    fn = np.count_nonzero(np.logical_and(y_pred == 0, y_true == 1))

    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0

    return round(accuracy, 4), round(precision, 4), round(recall, 4), round(f1, 4)

def cross_validation(x, y, num_folds=10, num_epochs=5000):
    hidden_size = 64
    learning_rate = 15e-4
    
    kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=1)
    acc_hist, prec_hist, rec_hist, f1_hist = [], [], [], []
    
    mean_tpr = 0.0
    mean_fpr = np.linspace(0, 1, 100)

    # --- Tracking the Best Fold ---
    best_acc = -1.0
    best_model = None
    best_x_train, best_y_train = None, None
    best_x_val, best_y_val = None, None

    for fold, (train_indices, val_indices) in enumerate(kf.split(x, y)):
        print(f"Fold {fold + 1}/{num_folds}")
        torch.manual_seed(0)

        x_train, y_train = x[train_indices].to(device), y[train_indices].to(device)
        x_val, y_val = x[val_indices].to(device), y[val_indices].to(device)

        model = FCN(input_size=x_train.shape[1], hidden_size=hidden_size, output_size=1).to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

        model.train()
        for epoch in range(num_epochs):
            optimizer.zero_grad()
            outputs = model(x_train)
            outputs = outputs.view(-1)
            y_train_float = y_train.type(torch.float)

            loss = criterion(outputs, y_train_float)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            outputs = model(x_val)
            outputs = outputs.view(-1)
            pred_probs = torch.sigmoid(outputs).cpu().numpy() 
            y_val_np = y_val.cpu().numpy()

        acc, prec, rec, f1 = plot_roc_curve(y_val_np, pred_probs)
        fpr, tpr, _ = roc_curve(y_val_np, pred_probs)
        
        mean_tpr += np.interp(mean_fpr, fpr, tpr)
        mean_tpr[0] = 0.0

        acc_hist.append(acc)
        prec_hist.append(prec)
        rec_hist.append(rec)
        f1_hist.append(f1)
        
        # --- Update Best Model ---
        if acc > best_acc:
            best_acc = acc
            best_model = copy.deepcopy(model)
            best_x_train, best_y_train = x_train.clone().cpu(), y_train.clone().cpu()
            best_x_val, best_y_val = x_val.clone().cpu(), y_val.clone().cpu()

    mean_tpr /= num_folds
    mean_tpr[-1] = 1.0

    return acc_hist, prec_hist, rec_hist, f1_hist, mean_tpr, best_model, best_x_train, best_y_train, best_x_val, best_y_val

In [ ]:
def evaluate_fcn_classification(dat_org, pos_label, neg_label, task_idx, num_epochs=5000):
    print(f"\n--- Running FCN Classification: {pos_label} vs {neg_label} ---")
    dat = dat_org.loc[dat_org.label.isin([pos_label, neg_label])].reset_index(drop=True)
    dat['case'] = [1 if ('f' in str(i)) else 0 for i in dat['case']]

    df_x_data = dat.drop(['label'], axis=1, errors='ignore')
    df_y_data = dat['label']

    ndf_x_data = df_x_data.copy()
    ndf_x_data[:] = minmax_scale(ndf_x_data)
    
    ndf_x_data = ndf_x_data.astype(float)
    data_train = torch.from_numpy(ndf_x_data.values).type(torch.float)
    
    targets_train = [0 if (i == pos_label) else 1 for i in df_y_data.tolist()]
    targets_train = torch.tensor(targets_train, dtype=torch.int64)

    acc, prec, rec, f1, mean_tpr, best_model, best_x_train, best_y_train, best_x_val, best_y_val = cross_validation(
        data_train, targets_train, num_folds=10, num_epochs=num_epochs
    )

    print("\nCV Average Results (Acc, Rec, Prec, F1):")
    print([round(np.mean(acc), 4), round(np.mean(rec), 4), round(np.mean(prec), 4), round(np.mean(f1), 4)])

    np.save(f"{task_idx}.interact_FCN.npy", mean_tpr)

    save_dir = "best_fold_fcn_data"
    os.makedirs(save_dir, exist_ok=True)
    
    torch.save(best_x_train, os.path.join(save_dir, f"{pos_label}_{neg_label}_x_train.pt"))
    torch.save(best_y_train, os.path.join(save_dir, f"{pos_label}_{neg_label}_y_train.pt"))
    torch.save(best_x_val, os.path.join(save_dir, f"{pos_label}_{neg_label}_x_val.pt"))
    torch.save(best_y_val, os.path.join(save_dir, f"{pos_label}_{neg_label}_y_val.pt"))
    torch.save(best_model.state_dict(), os.path.join(save_dir, f"{pos_label}_{neg_label}_model.pth"))
    
    return best_model, best_x_train, best_y_train, best_x_val, best_y_val

## 1. Import dataset and train a model

In [ ]:
# data: single_20.csv  /  interact_20.csv  / integrate_20.csv
# enc: raw / dot / solid
# loss: mem / spk
dat_org = pd.read_csv('.\\datasets\\integrate_20.csv') 

In [6]:
# 2.1 Intimacy vs non_Intimacy
best_model, best_x_train, best_y_train, best_x_val, best_y_val = evaluate_fcn_classification(
    dat_org, 'Intimacy', 'non_Intimacy', task_idx=1, num_epochs=5000
)


--- Running FCN Classification: Intimacy vs non_Intimacy ---
Fold 1/10
Fold 2/10
Fold 3/10
Fold 4/10
Fold 5/10
Fold 6/10
Fold 7/10
Fold 8/10
Fold 9/10
Fold 10/10

CV Average Results (Acc, Rec, Prec, F1):
[np.float64(0.8722), np.float64(0.8664), np.float64(0.8639), np.float64(0.8648)]
최고 성능의 폴드 데이터와 모델이 'best_fold_fcn_data/' 폴더에 저장되었습니다.


In [7]:
best_model, best_x_train, best_y_train, best_x_val, best_y_val = evaluate_fcn_classification(
    dat_org, 'Empathy', 'non_Empathy', task_idx=2, num_epochs=5000
)


--- Running FCN Classification: Empathy vs non_Empathy ---
Fold 1/10
Fold 2/10
Fold 3/10
Fold 4/10
Fold 5/10
Fold 6/10
Fold 7/10
Fold 8/10
Fold 9/10
Fold 10/10

CV Average Results (Acc, Rec, Prec, F1):
[np.float64(0.8886), np.float64(0.9076), np.float64(0.8841), np.float64(0.8955)]
최고 성능의 폴드 데이터와 모델이 'best_fold_fcn_data/' 폴더에 저장되었습니다.


In [8]:
best_model, best_x_train, best_y_train, best_x_val, best_y_val = evaluate_fcn_classification(
    dat_org, 'Competition', 'Cooperation', task_idx=3, num_epochs=5000
)


--- Running FCN Classification: Competition vs Cooperation ---
Fold 1/10
Fold 2/10
Fold 3/10
Fold 4/10
Fold 5/10
Fold 6/10
Fold 7/10
Fold 8/10
Fold 9/10
Fold 10/10

CV Average Results (Acc, Rec, Prec, F1):
[np.float64(0.8724), np.float64(0.8858), np.float64(0.8726), np.float64(0.8787)]
최고 성능의 폴드 데이터와 모델이 'best_fold_fcn_data/' 폴더에 저장되었습니다.


In [9]:
best_model, best_x_train, best_y_train, best_x_val, best_y_val = evaluate_fcn_classification(
    dat_org, 'High Focus', 'Low Focus', task_idx=4, num_epochs=5000
)


--- Running FCN Classification: High Focus vs Low Focus ---
Fold 1/10
Fold 2/10
Fold 3/10
Fold 4/10
Fold 5/10
Fold 6/10
Fold 7/10
Fold 8/10
Fold 9/10
Fold 10/10

CV Average Results (Acc, Rec, Prec, F1):
[np.float64(0.8885), np.float64(0.8738), np.float64(0.8962), np.float64(0.8846)]
최고 성능의 폴드 데이터와 모델이 'best_fold_fcn_data/' 폴더에 저장되었습니다.
